# Jev Generic Gym Continuous Controller

[Open in Colab](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_Generic_Gym_Controller_Colab.ipynb)

Generic adapter for any Gymnasium environment with a finite 1-D `Box` action space. No walker gait, PID, angle/speed threshold, or environment-specific recovery logic is used.

`Box actions -> local candidate lattice -> Jev Choice probabilities -> smooth continuous action -> reward/outcome feedback`.

Create the API key at **https://typesafe.ai** and save it in Colab Secrets as `TYPESAFE_API_KEY`.


In [ ]:
#@title 1. Install + configuration
!apt-get update -qq
!apt-get install -y -qq swig >/dev/null
!pip -q install "gymnasium[box2d]" requests imageio imageio-ffmpeg pillow

import os,time,json,getpass,itertools,math
from collections import deque
from pathlib import Path
import numpy as np, requests, gymnasium as gym, imageio.v2 as imageio
from IPython.display import Video,Image,display
from PIL import Image as PILImage

ENV_ID="BipedalWalker-v3"
OBJECTIVE="Maximize cumulative environment reward, avoid early termination, and prefer smooth useful actions."
OBS_NAMES=["hull_angle","hull_angular_velocity_scaled","horizontal_velocity_scaled","vertical_velocity_scaled",
"leg0_hip_angle","leg0_hip_speed","leg0_knee_angle_shifted","leg0_knee_speed","leg0_ground_contact",
"leg1_hip_angle","leg1_hip_speed","leg1_knee_angle_shifted","leg1_knee_speed","leg1_ground_contact",
*["lidar_"+str(i) for i in range(10)]]
ACT_NAMES=["leg0_hip_motor","leg0_knee_motor","leg1_hip_motor","leg1_knee_motor"]

MODEL="jev-latest"; API_URL="https://api.typesafe.ai/v1/systemone"
SEED=0; MAX_STEPS=1600; CONTROL_HORIZON=2; ACTION_STEP=.50
TEMP=.40; WINNER_WEIGHT=.45; MAX_DELTA=.40; HISTORY_LEN=10; MAX_CANDIDATES=243
VIDEO="/content/jev_generic.mp4"; GIF="/content/jev_generic.gif"; LOG="/content/jev_generic.json"

def api_key():
    k=None
    try:
        from google.colab import userdata
        k=userdata.get("TYPESAFE_API_KEY")
    except Exception: pass
    k=k or os.getenv("TYPESAFE_API_KEY")
    return k or getpass.getpass("TypeSafe API key from https://typesafe.ai: ").strip()
KEY=api_key()


In [ ]:
#@title 2. Generic action-space adapter + candidate lattice
class BoxAdapter:
    def __init__(self,a,o):
        if not isinstance(a,gym.spaces.Box) or len(a.shape)!=1: raise TypeError("finite 1-D Box action space required")
        self.a=a; self.o=o; self.lo=np.asarray(a.low,np.float32); self.hi=np.asarray(a.high,np.float32)
        if not np.all(np.isfinite(self.lo)) or not np.all(np.isfinite(self.hi)): raise ValueError("finite action bounds required")
        self.c=(self.lo+self.hi)/2; self.r=(self.hi-self.lo)/2; self.d=len(self.lo)
    def real(self,z): return np.clip(self.c+np.asarray(z)*self.r,self.lo,self.hi).astype(np.float32)
    def norm_obs(self,x):
        x=np.asarray(x,np.float32).reshape(-1)
        if not isinstance(self.o,gym.spaces.Box): return np.tanh(x)
        lo=np.asarray(self.o.low,np.float32).reshape(-1); hi=np.asarray(self.o.high,np.float32).reshape(-1)
        ok=np.isfinite(lo)&np.isfinite(hi)&(hi>lo); y=np.tanh(x)
        y[ok]=2*(x[ok]-lo[ok])/(hi[ok]-lo[ok])-1
        return np.clip(y,-5,5)

def rv(x,n=3): return [round(float(v),n) for v in np.asarray(x).reshape(-1)]

def lattice(cur,step,limit=243,seed=0):
    cur=np.asarray(cur,np.float32); d=len(cur); out=[]
    if 3**d<=limit:
        dirs=itertools.product((-1.,0.,1.),repeat=d)
    else:
        ds=[np.zeros(d)]
        for i in range(d):
            for s in (-1.,1.):
                q=np.zeros(d); q[i]=s; ds.append(q)
        rng=np.random.default_rng(seed)
        while len(ds)<limit:
            q=rng.normal(size=d); q/=max(np.linalg.norm(q),1e-9); ds.append(np.clip(q*math.sqrt(d),-1,1))
        dirs=ds
    seen=set()
    for q in dirs:
        q=np.asarray(q,np.float32); t=np.clip(cur+step*q,-1,1); k=tuple(np.round(t,5))
        if k not in seen: seen.add(k); out.append((q,t))
        if len(out)>=limit: break
    return out


In [ ]:
#@title 3. Generic Jev controller
class JevGenericContinuousController:
    def __init__(self,key,adapter,objective,obs_names=None,act_names=None):
        self.ad=adapter; self.objective=objective; self.on=obs_names; self.an=act_names
        self.cur=np.zeros(adapter.d,np.float32); self.hist=deque(maxlen=HISTORY_LEN); self.i=0
        self.s=requests.Session(); self.s.headers.update({"Authorization":f"Bearer {key}","Content-Type":"application/json"})
    def named(self,x,names,p):
        x=np.asarray(x).reshape(-1)
        return {str(names[i]) if names and i<len(names) else f"{p}_{i}":round(float(v),4) for i,v in enumerate(x)}
    def state(self,obs,prev,ret,step):
        obs=np.asarray(obs,np.float32).reshape(-1); no=self.ad.norm_obs(obs)
        d=np.zeros_like(obs) if prev is None else obs-np.asarray(prev,np.float32).reshape(-1)
        nd=np.zeros_like(no) if prev is None else no-self.ad.norm_obs(prev)
        h=[{"action":rv(x["action"]),"reward":round(x["reward"],4),"terminated":x["terminated"],"obs_delta_l2":round(x["delta"],4)} for x in self.hist]
        return {"environment":ENV_ID,"objective":self.objective,"step":step,"episode_return":round(float(ret),4),
                "observation":self.named(obs,self.on,"obs"),"observation_change":self.named(d,self.on,"delta"),
                "normalized_observation":self.named(no,self.on,"obs"),"normalized_change":self.named(nd,self.on,"delta"),
                "current_action":self.named(self.ad.real(self.cur),self.an,"action"),
                "recent_action_outcomes":h,
                "contract":"Choose a complete bounded continuous action. Higher Gym reward is better. Use recent outcomes as online evidence."}
    def candidates(self):
        m={}; c={}
        for j,(q,t) in enumerate(lattice(self.cur,ACTION_STEP,MAX_CANDIDATES,SEED+self.i)):
            k=f"a{j:03d}"; real=self.ad.real(t); m[k]=t
            c[k]=f"complete action normalized={rv(t)} real={rv(real)} relative_move={rv(q)}"
        return m,c
    def call(self,state,criteria):
        payload={"model":MODEL,"state":state,"questions":{"action":{"type":"choice",
            "instructions":"Choose the complete action most likely to improve the objective from the current state. Compare with recent action/reward outcomes. Avoid repeating actions associated with worse reward or termination.",
            "criteria":criteria}}}
        err=None
        for a in range(3):
            t=time.perf_counter()
            try:
                r=self.s.post(API_URL,json=payload,timeout=20); ms=(time.perf_counter()-t)*1000
                if r.status_code in (429,529) or r.status_code>=500: raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                r.raise_for_status(); z=r.json()["answers"]["action"]
                return z["choice"],float(z.get("confidence",0)),z.get("probabilities",{}),ms
            except Exception as e:
                err=e
                if a<2: time.sleep(2**a)
        raise RuntimeError(err)
    def decide(self,obs,prev,ret,step):
        m,c=self.candidates(); choice,conf,probs,ms=self.call(self.state(obs,prev,ret,step),c)
        ks=[k for k in m if float(probs.get(k,0))>0]
        if not ks:
            if choice not in m: raise RuntimeError("unknown Jev choice")
            target=m[choice].copy(); top=[(choice,1.)]
        else:
            p=np.array([max(float(probs[k]),1e-12) for k in ks]); p=p**(1/TEMP); p/=p.sum()
            mean=sum((w*m[k] for k,w in zip(ks,p)),np.zeros(self.ad.d))
            winner=m[choice] if choice in m else m[ks[int(np.argmax(p))]]
            target=np.clip((1-WINNER_WEIGHT)*mean+WINNER_WEIGHT*winner,-1,1).astype(np.float32)
            top=sorted(zip(ks,p.tolist()),key=lambda x:x[1],reverse=True)[:5]
        self.i+=1
        return {"target":target,"choice":choice,"confidence":conf,"latency_ms":ms,"top":top,"n":len(m)}
    def outcome(self,start,end,target,rewards,done):
        self.hist.append({"action":np.asarray(target).copy(),"reward":float(np.sum(rewards)),
                          "terminated":bool(done),"delta":float(np.linalg.norm(np.asarray(end)-np.asarray(start)))})


In [ ]:
#@title 4. Run — controller contains no environment-specific control rule
env=gym.make(ENV_ID,render_mode="rgb_array"); obs,_=env.reset(seed=SEED)
ad=BoxAdapter(env.action_space,env.observation_space)
ctl=JevGenericContinuousController(KEY,ad,OBJECTIVE,OBS_NAMES,ACT_NAMES)
print("action dimensions:",ad.d,"candidates:",len(lattice(np.zeros(ad.d),ACTION_STEP)))
ret=0.; step=0; prev_decision_obs=None; decisions=[]; frames=[]; gifs=[]; api_failures=0
writer=imageio.get_writer(VIDEO,format="FFMPEG",fps=12,codec="libx264",pixelformat="yuv420p",macro_block_size=1)
try:
    while step<MAX_STEPS:
        start_obs=np.asarray(obs).copy()
        try:
            d=ctl.decide(obs,prev_decision_obs,ret,step); target=d["target"].copy()
        except Exception as e:
            api_failures+=1; target=(.8*ctl.cur).astype(np.float32)
            d={"choice":"api_fallback","confidence":0.,"latency_ms":float("nan"),"top":[],"n":0}
        start=ctl.cur.copy(); rewards=[]; done=False
        for j in range(CONTROL_HORIZON):
            f=(j+1)/CONTROL_HORIZON; desired=(1-f)*start+f*target
            nxt=np.clip(ctl.cur+np.clip(desired-ctl.cur,-MAX_DELTA,MAX_DELTA),-1,1).astype(np.float32)
            act=ad.real(nxt); obs,r,term,trunc,_=env.step(act); done=term or trunc
            ctl.cur=nxt; ret+=float(r); rewards.append(float(r)); step+=1
            frames.append({"step":step,"action":rv(nxt),"reward":float(r),"return":ret})
            if step%4==0: writer.append_data(env.render())
            if step%8==0:
                im=PILImage.fromarray(env.render()).resize((450,300)); gifs.append(np.asarray(im))
            if done or step>=MAX_STEPS: break
        ctl.outcome(start_obs,obs,target,rewards,done); prev_decision_obs=start_obs
        decisions.append({"step":step,"choice":d["choice"],"confidence":d["confidence"],"latency_ms":d["latency_ms"],
                          "target":rv(target),"macro_reward":float(np.sum(rewards)),"return":ret,"top":d["top"]})
        if len(decisions)<=10 or len(decisions)%20==0 or done:
            print(f"decision {len(decisions):04d} | step {step:04d} | conf {d['confidence']:.3f} | {d['latency_ms']:.0f} ms | reward {np.sum(rewards):+.3f} | return {ret:+.2f} | action {rv(target,2)}")
        if done: print("episode ended:",step); break
finally:
    writer.close(); env.close()

if gifs: imageio.mimsave(GIF,gifs,duration=8/50,loop=0)
Path(LOG).write_text(json.dumps({"environment":ENV_ID,"objective":OBJECTIVE,"decisions":decisions,"frames":frames},indent=2))
print("\nRESULT | steps",step,"| return",round(ret,2),"| Jev decisions",len(decisions),"| API failures",api_failures)


In [ ]:
#@title 5. Display
display(Video(VIDEO,embed=True,html_attributes="controls loop"))
print("GIF fallback:"); display(Image(filename=GIF))
print("log:",LOG)


## Reuse it on another continuous Gym task

Change only `ENV_ID`, `OBJECTIVE`, and optionally `OBS_NAMES` / `ACT_NAMES`. The controller class remains unchanged.

Example: `ENV_ID="LunarLanderContinuous-v3"; OBS_NAMES=None; ACT_NAMES=None`.

This is a zero-training generic Jev adapter, not a pretrained locomotion policy. For strong real-time control, collect these Jev decisions and rewards, then distill them with RFDT into a local policy.
